In [1]:
import sys
!{sys.executable} -m pip install sacrebleu -q
!{sys.executable} -m pip install "transformers>=4.52.4" -q
!{sys.executable} -m pip install accelerate -q
!{sys.executable} -m pip install evaluate scikit-learn -q


In [2]:
import os
import sys
import json
import random
import re
import torch
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import snapshot_download
import evaluate

/dss/dsshome1/09/go34jos2/miniconda3/envs/qwen_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
direc = "/dss/dsshome1/09/go34jos2"
os.environ['Huggingface'] = os.path.join(direc, "huggingface")
os.environ['Para'] = os.path.join(direc, "para")
os.makedirs(os.environ['Huggingface'], exist_ok=True)
os.makedirs(os.environ['Para'], exist_ok=True)
lla = os.path.join(direc, "LLaMA-Factory")
data_output_dir = os.path.join(lla, "data")
train_src_file = os.path.join(direc, "train.de-hsb.de")
train_tgt_file = os.path.join(direc, "train.de-hsb.hsb")
dev_src_file = os.path.join(direc, "dev.de-hsb.de")
dev_tgt_file = os.path.join(direc, "dev.de-hsb.hsb")
train_file = os.path.join(data_output_dir, "train_data.json")
val_file = os.path.join(data_output_dir, "val_data.json")

In [4]:
direc = "/dss/dsshome1/09/go34jos2"
os.chdir(direc)
!git clone https://github.com/hiyouga/LLaMA-Factory.git
lla = os.path.join(direc, "LLaMA-Factory")
os.chdir(lla)
!pip install -e .[torch,metrics] -q
!pip install sacrebleu -q
os.chdir(direc)

Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 26012, done.
remote: Counting objects: 100% (401/401), done.
remote: Compressing objects: 100% (251/251), done.
remote: Total 26012 (delta 290), reused 150 (delta 150), pack-reused 25611 (from 4)
Receiving objects: 100% (26012/26012), 12.83 MiB | 30.69 MiB/s, done.
Resolving deltas: 100% (18617/18617), done.

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
ERROR: Package 'llamafactory' requires a different Python: 3.10.12 not in '>=3.11.0'

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip


In [5]:
#using regular expressions to clean up input corpora suggested by NRC
def preprocess_corpus(text):
    if not text:
      return ""
    text = re.sub(r'[\x01-\x09\x0b\x0c\x0e-\x1d\x7f]', '', text)
    text = re.sub(r'\x0d', '', text)
    text = re.sub(r'\t', ' ', text)
    text = re.sub(r'\\\\ ?[rn]', ' ', text)
    text = re.sub(r'\\ ?[rn]', ' ', text)
    text = re.sub(r' +', ' ', text)
    return text.strip()
#system prompt suggested by JUG and NRC
instruction_text = "Translate the following German text to Upper Sorbian.\nPut it in this format <hsb> Upper Sorbian translation </hsb>."

In [6]:
def constructing_dataset(src_file, tgt_file):
    data_list = []
    with open(src_file,encoding='utf-8') as f_src,open(tgt_file, encoding='utf-8') as f_tgt:
        for raw_src,raw_tgt in zip(f_src, f_tgt):
            preprocessed_src = preprocess_corpus(raw_src)
            preprocessed_tgt = preprocess_corpus(raw_tgt)
            if preprocessed_src and preprocessed_tgt:
                #construct sharegpt format
                human_value = f"{instruction_text}\n<deu> {preprocessed_src} </deu>"
                gpt_value = f"<hsb> {preprocessed_tgt} </hsb>"
                entry = {
                    "conversations":[
                        {"from": "human", "value": human_value},
                        {"from": "gpt", "value": gpt_value}
                    ]
                }
                data_list.append(entry)
    return data_list

In [7]:
train_data = constructing_dataset(train_src_file, train_tgt_file)
random.seed(1000)
random.shuffle(train_data)
val_data = constructing_dataset(dev_src_file, dev_tgt_file)
with open(train_file, 'w', encoding='utf-8') as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)
with open(val_file, 'w', encoding='utf-8') as f:
    json.dump(val_data, f, ensure_ascii=False, indent=2)

In [8]:
#register data
info_p =os.path.join(data_output_dir, "dataset_info.json")
with open(info_p, 'r', encoding='utf-8') as f:
    info = json.load(f)
info["dataset_train"] = {
    "file_name": "train_data.json",
    "formatting": "sharegpt",
    "columns": {"messages": "conversations"}
}
info["dataset_dev"] = {
    "file_name": "val_data.json",
    "formatting": "sharegpt",
    "columns": {"messages": "conversations"}
}
with open(info_p, 'w', encoding='utf-8') as f:
    json.dump(info, f, indent=2, ensure_ascii=False)

In [9]:
direc = "/dss/dsshome1/09/go34jos2"
os.chdir(direc)
local_model_dir = os.path.join(direc, "Qwen2.5-0.5B")
os.makedirs(local_model_dir, exist_ok=True)

In [10]:
#download model
model_name = "Qwen/Qwen2.5-0.5B"
snapshot_download(
    repo_id=model_name,
    local_dir=local_model_dir,
    local_dir_use_symlinks=False
)

/dss/dsshome1/09/go34jos2/miniconda3/envs/qwen_env/lib/python3.11/site-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
Fetching 10 files: 100%|██████████| 10/10 [00:02<00:00,  4.02it/s]


'/dss/dsshome1/09/go34jos2/Qwen2.5-0.5B'

In [11]:
os.environ["WORLD_SIZE"] = "1"
os.environ["RANK"] = "0"
os.environ["MASTER_ADDR"] = "127.0.0.1"
os.environ["MASTER_PORT"] = "29500"

In [12]:
direc = "/dss/dsshome1/09/go34jos2"
os.chdir(direc)
output_dir = os.path.join(direc,"qwen2p5_fft")
os.makedirs(output_dir, exist_ok=True)

In [ ]:
import os

user_home = "/dss/dsshome1/09/go34jos2"
python_executable = os.path.join(user_home, "miniconda3", "envs", "qwen_env", "bin", "python")
os.chdir(lla)
!{python_executable} -m llamafactory.cli train \
    --stage sft \
    --do_train \
    --model_name_or_path "{local_model_dir}" \
    --dataset dataset_train \
    --eval_dataset dataset_dev \
    --template chatml \
    --finetuning_type full \
    --output_dir "{output_dir}" \
    --overwrite_output_dir \
    --per_device_train_batch_size 2 \
    --per_device_eval_batch_size 2 \
    --gradient_accumulation_steps 32 \
    --learning_rate 7.0e-05 \
    --num_train_epochs 2.0 \
    --lr_scheduler_type cosine \
    --warmup_ratio 0.0 \
    --weight_decay 0.0 \
    --max_grad_norm 1.0 \
    --logging_steps 50 \
    --save_strategy "steps" \
    --save_steps 400 \
    --save_total_limit 2 \
    --eval_strategy "steps" \
    --eval_steps 400 \
    --plot_loss \
    --bf16 \
    --trust_remote_code True

os.chdir(direc)


[INFO|2026-01-18 16:19:21] llamafactory.hparams.parser:465 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: True, compute dtype: torch.bfloat16
[INFO|tokenization_utils_base.py:2093] 2026-01-18 16:19:21,773 >> loading file vocab.json
[INFO|tokenization_utils_base.py:2093] 2026-01-18 16:19:21,773 >> loading file merges.txt
[INFO|tokenization_utils_base.py:2093] 2026-01-18 16:19:21,773 >> loading file tokenizer.json
[INFO|tokenization_utils_base.py:2093] 2026-01-18 16:19:21,773 >> loading file added_tokens.json
[INFO|tokenization_utils_base.py:2093] 2026-01-18 16:19:21,773 >> loading file special_tokens_map.json
[INFO|tokenization_utils_base.py:2093] 2026-01-18 16:19:21,773 >> loading file tokenizer_config.json
[INFO|tokenization_utils_base.py:2093] 2026-01-18 16:19:21,773 >> loading file chat_template.jinja
[INFO|tokenization_utils_base.py:2364] 2026-01-18 16:19:22,093 >> Special tokens have been added in the vocabulary, make sure the associated word embeddings a

In [ ]:
user_home = "/dss/dsshome1/09/go34jos2"
output_dir = os.path.join(user_home, "qwen2p5_fft")
checkpoint_dir = os.path.join(output_dir, "checkpoint-5600")
tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
if tokenizer.chat_template is None:
     tokenizer.chat_template = "{% if not add_generation_prompt is defined %}{% set add_generation_prompt = false %}{% endif %}{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
model = AutoModelForCausalLM.from_pretrained(
    checkpoint_dir,
    device_map="auto",
    dtype=torch.bfloat16
)
#extract hsb
def extract_translation(text):
    match = re.search(r"<hsb>\s*(.*?)\s*</hsb>", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return text.strip()
with open(dev_src_file,encoding='utf-8') as f:
    src_lines = [preprocess_corpus(line) for line in f]
predictions = []
batch_size = 16
for i in tqdm(range(0, len(src_lines), batch_size)):
    batch_src = src_lines[i:i+batch_size]
    batch_inputs = []
    for src in batch_src:
        user_content = instruction_text + "\n<deu> " + src + " </deu>"
        messages = [
            {"role": "user", "content": user_content}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        batch_inputs.append(text)
    inputs = tokenizer(batch_inputs, return_tensors="pt", padding=True, truncation=True).to(model.device)
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id
        )
    generated_ids = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    raw_preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    clean_preds = [extract_translation(p) for p in raw_preds]
    predictions.extend(clean_preds)

In [ ]:
with open(dev_tgt_file,encoding='utf-8') as f:
    references = [preprocess_corpus(line) for line in f]
sacrebleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")
bleu_result = sacrebleu.compute(predictions=predictions, references=[[r] for r in references])
chrf_result = chrf.compute(predictions=predictions, references=[[r] for r in references], word_order=2)
print(f"SacreBLEU: {bleu_result['score']:.2f}")
print(f"chrF++:    {chrf_result['score']:.2f}")